<a href="https://colab.research.google.com/github/alexiachavez-sketch/build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/ELECTRONIT_Multi_Agent_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install required libraries
!pip install -q langgraph langchain-core langchain-openai

import os
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from google.colab import userdata

# Load your API key securely from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize LLM model (swapped from Azure to ChatOpenAI)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

Step 2

In [10]:
ORDERS_DB = {
    "ORD-1001": {"product": "AlphaBook Pro", "quantity": 1, "status": "Shipped", "delivery_date": "2026-10-22"},
    "ORD-1002": {"product": "GammaAir X", "quantity": 2, "status": "Processing", "delivery_date": "2026-10-25"},
    "ORD-1003": {"product": "SpectraBook S", "quantity": 1, "status": "Delivered", "delivery_date": "2026-10-15"},
}

@tool
def lookup_order_status(order_id: str) -> str:
    """Look up order status and shipping details by order ID."""
    order = ORDERS_DB.get(order_id.upper())
    if order:
        return (f"Order ID: {order_id.upper()}\n"
                f"Product: {order['product']}\n"
                f"Quantity: {order['quantity']}\n"
                f"Status: {order['status']}\n"
                f"Delivery Date: {order['delivery_date']}")
    return f"Order ID '{order_id}' was not found. Please verify the order ID."

step 2

In [11]:
CATALOG_DB = [
    {"name": "AlphaBook Pro", "price": "$1,499", "specs": "12th Gen Intel i7, 16GB RAM, 512GB SSD. Great for everyday work."},
    {"name": "GammaAir X", "price": "$1,399", "specs": "AMD Ryzen 7, 32GB RAM, 512GB SSD. Lightweight & portable."},
    {"name": "SpectraBook S", "price": "$2,499", "specs": "Intel Core i9, 64GB RAM, 2TB SSD. High performance workstation."},
    {"name": "OmegaPro G17", "price": "$2,199", "specs": "Ryzen 9 5900HX, 32GB RAM, 1TB SSD. Gaming laptop."},
]

@tool
def get_product_catalog(query: str = "") -> str:
    """Retrieve available laptops, specifications, and pricing from ELECTRONIT."""
    results = [f"- {item['name']} ({item['price']}): {item['specs']}" for item in CATALOG_DB]
    return "\n".join(results)

step 3

In [12]:
# 1. Main Role Prompt
MAIN_STORE_PROMPT = """You are a friendly customer service assistant for ELECTRONIT, an electronics store.
Be concise, professional, and helpful. Never invent order details or specs. Escalate when unsure."""

# 2. Order Agent Prompt
ORDER_AGENT_PROMPT = f"""{MAIN_STORE_PROMPT}
You are the Order Status Agent. Help customers track purchases, check status, and get delivery updates using the `lookup_order_status` tool."""

# 3. Refund / Return Policy Agent Prompt
REFUND_AGENT_PROMPT = f"""{MAIN_STORE_PROMPT}
You are the Refund & Return Policy Agent for ELECTRONIT.
Rules:
- Returns/Exchanges are accepted within 30 days of delivery.
- Opened items have a 15% restocking fee unless defective.
- Refunds take 5-7 business days to process.
- Return shipping is free for defective items."""

# 4. Product Recommendation Agent Prompt
PRODUCT_RECOMMENDER_PROMPT = f"""{MAIN_STORE_PROMPT}
You are the Product Recommendation Agent.
Help customers choose electronics based on needs, budget, and specs using the `get_product_catalog` tool."""

# 5. Router Supervisor Prompt
ROUTER_PROMPT = """Classify the latest user message into EXACTLY ONE label:
- ORDER: Order tracking, delivery, or shipping queries.
- REFUND: Return eligibility, policy, or refund timeline questions.
- RECOMMENDATION: Suggestions, product specs, or comparisons.
- FALLBACK: Off-topic or non-ELECTRONIT queries.

Output ONLY ONE WORD: ORDER, REFUND, RECOMMENDATION, or FALLBACK."""

Step 4: Define Graph State & Agent Nodes

In [13]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# Router Node
def router_node(state: State):
    messages = [SystemMessage(content=ROUTER_PROMPT)] + state["messages"]
    return {"messages": [model.invoke(messages)]}

def route_next(state: State) -> str:
    last_msg = state["messages"][-1].content.strip().upper()
    if "ORDER" in last_msg:
        return "order_agent"
    elif "REFUND" in last_msg:
        return "refund_agent"
    elif "RECOMMENDATION" in last_msg:
        return "product_agent"
    else:
        return "fallback_node"

# Agent Nodes
def order_agent_node(state: State):
    llm_tools = model.bind_tools([lookup_order_status])
    messages = [SystemMessage(content=ORDER_AGENT_PROMPT)] + state["messages"][:-1]
    res = llm_tools.invoke(messages)
    if res.tool_calls:
        tool_call = res.tool_calls[0]
        tool_output = lookup_order_status.invoke(tool_call["args"])
        final_msgs = messages + [res, ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"])]
        return {"messages": [model.invoke(final_msgs)]}
    return {"messages": [res]}

def refund_agent_node(state: State):
    messages = [SystemMessage(content=REFUND_AGENT_PROMPT)] + state["messages"][:-1]
    return {"messages": [model.invoke(messages)]}

def product_agent_node(state: State):
    llm_tools = model.bind_tools([get_product_catalog])
    messages = [SystemMessage(content=PRODUCT_RECOMMENDER_PROMPT)] + state["messages"][:-1]
    res = llm_tools.invoke(messages)
    if res.tool_calls:
        tool_call = res.tool_calls[0]
        tool_output = get_product_catalog.invoke(tool_call["args"])
        final_msgs = messages + [res, ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"])]
        return {"messages": [model.invoke(final_msgs)]}
    return {"messages": [res]}

def fallback_node(state: State):
    return {"messages": [AIMessage(content="I’m sorry, but I can only assist with questions about orders, returns, refunds, and product recommendations at ELECTRONIT. Is there anything related to our store I can assist with?")]}

Step 5: Compile Graph with MemorySaver

In [14]:
builder = StateGraph(State)

# Add Nodes
builder.add_node("router", router_node)
builder.add_node("order_agent", order_agent_node)
builder.add_node("refund_agent", refund_agent_node)
builder.add_node("product_agent", product_agent_node)
builder.add_node("fallback_node", fallback_node)

# Entry point
builder.set_entry_point("router")

# Add conditional routing
builder.add_conditional_edges(
    "router",
    route_next,
    {
        "order_agent": "order_agent",
        "refund_agent": "refund_agent",
        "product_agent": "product_agent",
        "fallback_node": "fallback_node",
    }
)

# Connect to END
builder.add_edge("order_agent", END)
builder.add_edge("refund_agent", END)
builder.add_edge("product_agent", END)
builder.add_edge("fallback_node", END)

# Attach checkpointer memory
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [7]:
from langgraph.prebuilt import ToolNode, tools_condition

# 1. Define tools
tools = [get_product_info, check_order_status]
model_with_tools = model.bind_tools(tools)

# 2. Define Assistant node
def call_model(state: AgentState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

# 3. Build graph with tool execution support
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools)) # Node to execute tools

workflow.set_entry_point("agent")

# Add conditional edge: if agent calls a tool -> go to "tools", else -> END
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("Phase 5 Multi-Agent Chatbot graph (with tools) compiled successfully!")

Phase 5 Multi-Agent Chatbot graph (with tools) compiled successfully!


Step 6: Test Persistent Conversation Memory

In [16]:
# Pass thread_id in config to keep context across multiple turns
config = {"configurable": {"thread_id": "customer-001"}}

def run_chat(user_input: str):
    print(f"\nUSER: {user_input}")
    res = graph.invoke({"messages": [HumanMessage(content=user_input)]}, config)
    print(f"BOT: {res['messages'][-1].content}")

# Multi-turn conversation test
run_chat("Hi, where is order ORD-1001?")
run_chat("When will it arrive?")  # Relies on memory of ORD-1001 from turn 1


USER: Hi, where is order ORD-1001?
BOT: Your order ORD-1001 for the AlphaBook Pro has been shipped. It is expected to arrive on October 22, 2026. If you have any further questions, feel free to ask!

USER: When will it arrive?
BOT: Your order ORD-1001 for the AlphaBook Pro is expected to arrive on October 22, 2026. If you have any other questions, feel free to ask!


In [17]:

def run_test_turn(user_input: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n[Thread: {thread_id}] 👤 USER: {user_input}")
    output = graph.invoke({"messages": [HumanMessage(content=user_input)]}, config)
    print(f"[Thread: {thread_id}] 🤖 BOT:  {output['messages'][-1].content}")

# ==============================================================================
# TEST 1: 3-Turn Conversation Proving Memory & Agent Switching
# ==============================================================================
print("=" * 60)
print("TEST 1: MULTI-TURN CONVERSATION WITH MEMORY & AGENT SWITCHING")
print("=" * 60)

THREAD_1 = "customer-001"

# Turn 1: Check Order Status (Order Status Agent)
run_test_turn("Hi, I want to check order ORD-1001.", THREAD_1)

# Turn 2: Follow-up question relying on memory (Order Status Agent)
run_test_turn("When will it arrive?", THREAD_1)

# Turn 3: Switch to Refund Agent while retaining context (Refund Policy Agent)
run_test_turn("Can I return it if I don't like it?", THREAD_1)


# ==============================================================================
# TEST 2: Individual Agent & Edge Case Tests
# ==============================================================================
print("\n" + "=" * 60)
print("TEST 2: INDIVIDUAL AGENTS & EDGE CASES")
print("=" * 60)

# Product Recommendation Agent Test
run_test_turn("I need a laptop for high-end video editing and 3D rendering. What do you recommend?", "customer-002")

# Edge Case A: Invalid Order ID
run_test_turn("Can you give me status on order ORD-9999?", "customer-003")

# Edge Case B: Off-topic Question (Fallback Node)
run_test_turn("Who won the last Super Bowl?", "customer-004")

# Edge Case C: Vague Request
run_test_turn("I need help with something.", "customer-005")


# ==============================================================================
# TEST 3: Memory Isolation Proof
# ==============================================================================
print("\n" + "=" * 60)
print("TEST 3: MEMORY ISOLATION PROOF")
print("=" * 60)

THREAD_2 = "customer-002-isolated"

# Query Thread 2 about the order mentioned in Thread 1
run_test_turn("When will my order arrive?", THREAD_2)

TEST 1: MULTI-TURN CONVERSATION WITH MEMORY & AGENT SWITCHING

[Thread: customer-001] 👤 USER: Hi, I want to check order ORD-1001.
[Thread: customer-001] 🤖 BOT:  Your order ORD-1001 for the AlphaBook Pro has been shipped. The expected delivery date is October 22, 2026. If you need further assistance, just let me know!

[Thread: customer-001] 👤 USER: When will it arrive?
[Thread: customer-001] 🤖 BOT:  Your order ORD-1001 for the AlphaBook Pro is expected to arrive on October 22, 2026. If you have any more questions, feel free to ask!

[Thread: customer-001] 👤 USER: Can I return it if I don't like it?
[Thread: customer-001] 🤖 BOT:  Yes, you can return the item within 30 days of delivery if you don't like it. Please note that opened items will incur a 15% restocking fee unless the item is defective. If you need assistance with the return process, let me know!

TEST 2: INDIVIDUAL AGENTS & EDGE CASES

[Thread: customer-002] 👤 USER: I need a laptop for high-end video editing and 3D rendering.